
### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [ ]:
import os
from dotenv import load_dotenv

# Set override=True so that updating the .env file is immediately loaded without kernel restarts
load_dotenv(override=True)

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")



### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash-lite",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [4]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='48ea537a-ef99-494c-b300-601b1a4548a4'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1150-86be-76e1-b84a-1ba98340f99a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='48ea537a-ef99-494c-b300-601b1a4548a4'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1150-86be-76e1-b84a-1ba98340f99a-0', tool_calls=[], invalid_tool_calls=[], us

## Token Size

In [16]:
import os
from dotenv import load_dotenv

# Set override=True so that updating the .env file is immediately loaded without kernel restarts
load_dotenv(override=True)

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [18]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~96 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='bc17d6dd-f417-4144-82fc-dce8e689eda9'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'1b34e23f-b36e-45c3-9462-f272fcb2b238': 'CpMCAQw51se4zwByZOxtQpWCyWL/oQ1gVx6TXTFMEAlidOyz1opgDXDR9a1pNtNpS9v+IO7ykONnZ9Ts56Vrnz6zZfHKNvj3j7Wy5BU9XmKvvNQSGBW7fayNwB8dLeEuPY4bekOZYeAvomgRV/Rv4qbYs3BAiMvgAsMJ8jovZAgv2/zADYh/nFzgL06lSmUGubalKpqPNgN4ZTJFC7WtEQEqJrEIhkLLG5k9KI+qElC9xGtTtOWMknUEd4SD7u975Qa6wWxjcyzF2+tW7W/o75Fvm/GByGcTpHgoKJBia71tFBY9qY9Iz7zXRVgPmX+CiALIrdq42qURdQuHMQb7y9RC6FmXpYQ+3g3F2gR3StdYtwHtCiA='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1161-8b43-7e71-85ea-a647cfdeabea-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}

KeyboardInterrupt: 

fraction

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="google_genai:gemini-2.5-flash",  # Make sure the prefix is here
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",  # Make sure the prefix is here too
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)


config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~36 tokens (0.0281%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='fc06f78f-37f3-47e8-968d-2de970cda8fd'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'79207742-ebfd-4adb-94e1-6b05b6dd027e': 'CrUBAQw51sd5aOPJgU3A1yywXjtMO6JYrTg9Ni3xxKlpKs0IucklZxucfvlUGZsV3rwwZP093jKGGz1rjlWSy0eZ9LthYARnt1QGXJRVoRxPTy4o7mcLhlt3jU4HdjHxsDC034zlmRSCI0pZgF/BS4U8CGIK232F/FyRdvCLHYcr/+u/ygH2tDM9i/4mMggAWF+D+Cc0HZeeiFzCNYRpy5DAC64ou0s4dlfay2/5T+Ij3TRMCodb8A=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1165-f78c-7fe0-a7d3-25028a0cbca6-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '79207742-ebfd-4adb-94e1-6b05b6dd027e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens

### HUMAN IN LOOP MIDDLEWARE


In [21]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [22]:

agent=create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [23]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='a508f7f7-6e51-4386-bea5-0823b22feda8'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'93f55d7a-1253-46e1-a2ee-fab2fc05afcc': 'CqcEAQw51seMnUptszRcZL95TMgMfTMiPxw4piKbqrqXO4rtyQ2ClIV4BLkLi5jvPcFyKIXIUbLDXS74NgUrJLQbrMorZmeCkG4XBgIRjSAGrtVrYSOiUMEg/ZvLEUZ7Tei6p4VxGrVWwBzdOji/Q0/f5l0MeZDsMWnM05Opci4SmcgDRHGBPgN9u9kid4SV+uDCHuAFCcWKksmc5wLHwMxd7lj+aqzME6ftyq4bR5g9OJZAFEC+CojS8vpiN1iZZh8cKZzEvHUpjJpgWj78UQkD5vNZYd1DUjWX9gzeri6N2gMHrZTOQgxjp5hQfzl6pSpzaBS0ZpbLI2vOYNcCJuutsj9j6cyUsDYsnSKRwF0iX+vtURU29F3CW75/7eNYPOc8oUjdHb2Hny59WbwFoIY5K2HGkjX/GZBuO8VzuDeN2OdlcGQDdAnUQHSomA8dL92t/9b9y+Hkrk1TBCLkzNCw8qMRyzjNMKePBTaJPzwtoMuY4nVbe2uPXBpxV3GiPqkNrZUfocRkZbGzvWw5

In [25]:

from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I have sent an email to john@test.com with the subject 'Hello' and body 'How are you?'.


Reject


In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [29]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [32]:

# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused!")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

In [33]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='9f6780b9-b38a-4924-8e08-c2a3085bbba0'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "body": "How are you?", "subject": "Hello"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f116d-d0a5-7de0-98a7-92e30aa05255-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'body': 'How are you?', 'subject': 'Hello'}, 'id': '59350adc-5f3f-4b59-80e1-291de65653d4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 33, 'total_tokens': 165, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='User rejected the tool call for `send_email_tool` with id 59350ad

Edit    

In [35]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [36]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [37]:

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='3cbd9bca-b5cf-494f-a1a6-0a51cf16ef92'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "wrong@email.com", "body": "Hello", "subject": "Test"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f116e-cef1-7201-ae39-241c39570e61-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'body': 'Hello', 'subject': 'Test'}, 'id': 'e02181df-c326-4b2e-bdcd-cf3d27a08eba', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 31, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}})],
 '__interrupt__': [Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'recipient':

In [38]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: Email sent to correct@email.com with subject 'Corrected Subject'.


In [39]:

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='3cbd9bca-b5cf-494f-a1a6-0a51cf16ef92'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "wrong@email.com", "body": "Hello", "subject": "Test"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f116e-cef1-7201-ae39-241c39570e61-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': 'e02181df-c326-4b2e-bdcd-cf3d27a08eba'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 31, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="Email sent to correct@email.com wi